# 6.11 LightGBM

**Machine Learning Mastery: Mathematics to MLOps** · Chapter 6: Classification Algorithms

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/RithikRaj129/ml-mastery/blob/main/chapter-06-classification-algorithms/06.11_lightgbm.ipynb)

Companion code for Section 6.11 of the book. Run the cells in order from top to bottom.

**Example 6.11.1**

In [ ]:
import lightgbm as lgb
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.datasets import load_breast_cancer
from sklearn.metrics import roc_auc_score, classification_report
import numpy as np
import matplotlib.pyplot as plt

data = load_breast_cancer()
X, y = data.data, data.target
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2,
                                                      stratify=y, random_state=42)

# ── LightGBM with early stopping ─────────────────────────────────
lgb_train = lgb.Dataset(X_train, label=y_train)
lgb_val   = lgb.Dataset(X_test,  label=y_test,  reference=lgb_train)

params = {
    'objective':        'binary',
    'metric':           'binary_logloss',
    'boosting_type':    'gbdt',
    'num_leaves':       31,
    'learning_rate':    0.05,
    'feature_fraction': 0.8,     # Fraction of features per tree
    'bagging_fraction': 0.8,     # Fraction of samples per tree
    'bagging_freq':     5,
    'reg_alpha':        0.1,
    'reg_lambda':       1.0,
    'verbose':         -1
}

callbacks = [lgb.early_stopping(50), lgb.log_evaluation(period=100)]
model_lgb = lgb.train(params, lgb_train, num_boost_round=500,
                       valid_sets=[lgb_train, lgb_val],
                       callbacks=callbacks)

print(f'Best iteration: {model_lgb.best_iteration}')
y_pred_lgb = (model_lgb.predict(X_test) >= 0.5).astype(int)
y_prob_lgb = model_lgb.predict(X_test)
acc = (y_pred_lgb == y_test).mean()
auc = roc_auc_score(y_test, y_prob_lgb)
print(f'Test Accuracy: {acc:.4f}  AUC: {auc:.4f}')

# ── Sklearn-compatible API ────────────────────────────────────────
lgb_sk = lgb.LGBMClassifier(
    n_estimators=300, learning_rate=0.05, num_leaves=31,
    feature_fraction=0.8, bagging_fraction=0.8, bagging_freq=5,
    reg_alpha=0.1, reg_lambda=1.0, random_state=42, verbose=-1
)
lgb_sk.fit(X_train, y_train)
print(f'\nLGBM Sklearn API: {lgb_sk.score(X_test, y_test):.4f}')

# ── LightGBM-specific hyperparameters ────────────────────────────
print('\n=== LightGBM Key Parameters ===')
lgb_params = {
    'num_leaves':       '31 (default); max tree leaves; increase for complex patterns',
    'max_depth':        '-1 (unlimited); use num_leaves to control complexity instead',
    'min_data_in_leaf': '20 (default); minimum samples per leaf; prevents overfitting',
    'feature_fraction': '0.8; fraction of features sampled per tree (like colsample_bytree)',
    'bagging_fraction': '0.8; fraction of data sampled per tree (like subsample)',
    'learning_rate':    '0.05-0.1; lower needs more n_estimators',
    'lambda_l1/l2':     'L1/L2 regularisation; equivalent to reg_alpha/lambda in XGBoost',
}
for p, desc in lgb_params.items():
    print(f'  {p:<22}: {desc}')